In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import (
    avg,
    stddev,
    current_timestamp
)

sales = spark.table(
    "agentdb.gold.fact_sales"
)

inventory = spark.table(
    "agentdb.gold.snapshot_inventory_health"
)

supplier_risk = spark.table(
    "agentdb.gold.snapshot_supplier_risk"
)

w7 = Window.partitionBy(
    "product_key",
    "store_key"
).orderBy(
    "calendar_key"
).rowsBetween(-6, 0)

w30 = Window.partitionBy(
    "product_key",
    "store_key"
).orderBy(
    "calendar_key"
).rowsBetween(-29, 0)

forecast_features = (
    sales

    .withColumn(
        "sales_7d_avg",
        avg("sales_qty").over(w7)
    )

    .withColumn(
        "sales_30d_avg",
        avg("sales_qty").over(w30)
    )

    .withColumn(
        "sales_7d_stddev",
        stddev("sales_qty").over(w7)
    )

    .withColumn(
        "sales_30d_stddev",
        stddev("sales_qty").over(w30)
    )

    .join(
        inventory.select(
            "product_key",
            "store_key",
            "inventory_qty",
            "days_of_cover"
        ),
        ["product_key", "store_key"],
        "left"
    )

    .withColumn(
        "created_at",
        current_timestamp()
    )
)

forecast_features.write \
    .mode("overwrite") \
    .saveAsTable(
        "agentdb.reporting.rpt_forecast_features"
    )

In [0]:
from pyspark.sql.functions import (
    when,
    current_timestamp
)

inventory = spark.table(
    "agentdb.gold.snapshot_inventory_health"
)

stockout_risk = (

    inventory

    .withColumn(
        "stockout_probability",

        when(
            inventory.days_of_cover < 7,
            0.95
        )
        .when(
            inventory.days_of_cover < 14,
            0.70
        )
        .when(
            inventory.days_of_cover < 30,
            0.40
        )
        .otherwise(
            0.10
        )
    )

    .withColumn(
        "risk_level",

        when(
            inventory.days_of_cover < 7,
            "CRITICAL"
        )
        .when(
            inventory.days_of_cover < 14,
            "HIGH"
        )
        .when(
            inventory.days_of_cover < 30,
            "MEDIUM"
        )
        .otherwise(
            "LOW"
        )
    )

    .withColumn(
        "forecast_demand_7d",
        inventory.inventory_qty / inventory.days_of_cover * 7
    )

    .withColumn(
        "created_at",
        current_timestamp()
    )
)

stockout_risk.select(
    "product_key",
    "store_key",
    "inventory_qty",
    "forecast_demand_7d",
    "days_of_cover",
    "stockout_probability",
    "risk_level",
    "created_at"
).write \
    .mode("overwrite") \
    .saveAsTable(
        "agentdb.reporting.rpt_stockout_risk"
    )

In [0]:
from pyspark.sql.functions import (
    current_timestamp,
    when,
    lit
)

supplier_snapshot = spark.table(
    "agentdb.gold.snapshot_supplier_risk"
)

supplier_risk = (

    supplier_snapshot

    .withColumn(
        "supplier_risk_level",

        when(
            supplier_snapshot.risk_score > 0.8,
            "CRITICAL"
        )
        .when(
            supplier_snapshot.risk_score > 0.6,
            "HIGH"
        )
        .when(
            supplier_snapshot.risk_score > 0.3,
            "MEDIUM"
        )
        .otherwise(
            "LOW"
        )
    )

    .withColumn(
        "created_at",
        current_timestamp()
    )
)

supplier_risk.select(
    "supplier_key",
    "risk_score",
    lit(0).alias("disruption_count"),
    lit(0.0).alias("avg_delay_days"),
    lit(0).alias("open_purchase_orders"),
    lit(0).alias("impacted_products"),
    "supplier_risk_level",
    "created_at"
).write \
    .mode("overwrite") \
    .saveAsTable(
        "agentdb.reporting.rpt_supplier_risk"
    )

In [0]:
from pyspark.sql.functions import (
    current_timestamp,
    when,
    greatest,
    lit
)

inventory = spark.table(
    "agentdb.gold.snapshot_inventory_health"
)

candidates = (

    inventory

    .filter(
        inventory.days_of_cover < 37
    )

    .withColumn(
        "reorder_point",
        lit(38.0)
    )

    .withColumn(
        "recommended_order_qty",

        greatest(
            lit(0),
            (38 - inventory.days_of_cover)
            * inventory.inventory_qty
            / inventory.days_of_cover
        )
    )

    .withColumn(
        "urgency_score",

        when(
            inventory.days_of_cover < 36,
            100.0
        )
        .otherwise(
            50.0
        )
    )

    .withColumn(
        "recommendation_type",
        lit("REORDER")
    )

    .withColumn(
        "created_at",
        current_timestamp()
    )
)

candidates.select(
    "product_key",
    "store_key",
    lit(None).cast("long").alias("dc_key"),
    "inventory_qty",
    "reorder_point",
    "recommended_order_qty",
    "urgency_score",
    "recommendation_type",
    "created_at"
).write \
    .mode("overwrite") \
    .saveAsTable(
        "agentdb.reporting.rpt_replenishment_candidates"
    )

In [0]:
from pyspark.sql.functions import (
    current_date,
    current_timestamp,
    sum,
    count
)

sales = spark.table(
    "agentdb.gold.fact_sales"
)

inventory = spark.table(
    "agentdb.gold.fact_inventory_snapshot"
)

purchase_orders = spark.table(
    "agentdb.gold.fact_purchase_orders"
)

supplier_risk = spark.table(
    "agentdb.reporting.rpt_supplier_risk"
)

daily_status = (

    sales.agg(
        sum("sales_qty").cast("double").alias("total_sales")
    )

    .crossJoin(
        inventory.agg(
            sum("inventory_qty")
            .alias("total_inventory")
        )
    )

    .crossJoin(
        purchase_orders.agg(
            count("*")
            .alias("total_open_purchase_orders")
        )
    )

    .crossJoin(
        supplier_risk.filter(
            supplier_risk.supplier_risk_level.isin(
                "HIGH",
                "CRITICAL"
            )
        ).agg(
            count("*")
            .alias("suppliers_at_risk")
        )
    )

    .withColumn(
        "snapshot_date",
        current_date()
    )

    .withColumn(
        "total_disruptions",
        lit(1).cast("long")
    )

    .withColumn(
        "stores_at_risk",
        lit(0).cast("long")
    )

    .withColumn(
        "products_at_risk",
        lit(0).cast("long")
    )

    .withColumn(
        "created_at",
        current_timestamp()
    )
)

daily_status.write \
    .mode("overwrite") \
    .saveAsTable(
        "agentdb.reporting.rpt_daily_supply_chain_status"
    )

In [0]:
tables = [row.tableName for row in spark.sql("SHOW TABLES IN agentdb.reporting").collect()]
counts = [(t, spark.table(f"agentdb.reporting.{t}").count()) for t in tables]
df = spark.createDataFrame(counts, ["table_name", "row_count"])
display(df)